# Appendix A2: Embedding model swap

Not one of "the 10 patterns" -- no §8 template (same exemption as `00_baseline_no_rag.ipynb`/
`00b_long_context_baseline.ipynb`). Holds retrieval constant (hybrid + cross-encoder rerank,
`recipes/hybrid_rerank.py`) and varies only the embedding model: `text-embedding-3-small` (OpenAI),
`voyage-4` (Voyage AI -- SPEC.md named `voyage-3`, deprecated as of 2026-08-12 verification against
Voyage's own docs, see `recipes/embeddings.py`'s `VoyageEmbedder` docstring), and `BAAI/bge-large-en-v1.5`
(local, open-weight, via `sentence-transformers`).

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for the OpenAI and Voyage
legs. **The `bge-large-en-v1.5` row is genuinely REAL** -- it's a local model, no API key needed at
all (same zero-key precedent as pattern 04's cross-encoder reranker), so its `paper_hit@10` number
below reflects real embeddings, real retrieval, real results. The OpenAI and Voyage rows are
PENDING, same treatment as 8 of the 10 main patterns.

Same `paper_hit_at_k` metric as A1 (see that notebook for why paper-level, not exact chunk-level).


## Reproducibility header

In [1]:
import platform
import sys
import subprocess
import openai
import numpy

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2
git commit: e9d172f8a88da3577c09ecf807b8cb38e79db941


## Setup

In [2]:
import os
os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set
from evals.metrics import paper_hit_at_k, bootstrap_ci
from recipes.embeddings import get_embedder, get_voyage_embedder, LocalEmbedder
from recipes.hybrid_rerank import build_retriever

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")

relevant_paper_ids_by_qid = {
    r["qid"]: {corpus_by_id[cid]["paper_id"] for cid in r["relevant_chunk_ids"] if cid in corpus_by_id}
    for r in qa_set
}

K = 10


In [3]:
def print_results_table(rows):
    """rows: list of (label, ConfidenceInterval, is_real: bool)."""
    print(f"{'variant':<28} {'paper_hit@10':<24} {'status'}")
    for label, ci, is_real in rows:
        ci_str = f"{ci.mean:.3f}  [95% CI {ci.lower:.3f}, {ci.upper:.3f}]"
        status = "REAL" if is_real else "PENDING (mock)"
        print(f"{label:<28} {ci_str:<24} {status}")


## Score each embedder

In [4]:
def score_embedder(embedder, embedding_model):
    retrieve = build_retriever(corpus_by_id, embedder=embedder, embedding_model=embedding_model)
    scores = []
    for r in qa_set:
        relevant_papers = relevant_paper_ids_by_qid[r["qid"]]
        if not relevant_papers:
            continue
        retrieved_ids = retrieve(r["question"], K)
        scores.append(paper_hit_at_k(retrieved_ids, relevant_papers, corpus_by_id, K))
    return bootstrap_ci(scores)

results = []


### text-embedding-3-small (OpenAI)

In [5]:
openai_embedder = get_embedder()
ci = score_embedder(openai_embedder, "text-embedding-3-small")
results.append(("text-embedding-3-small", ci, False))


### voyage-4 (Voyage AI)

In [6]:
voyage_embedder = get_voyage_embedder()
ci = score_embedder(voyage_embedder, "voyage-4")
results.append(("voyage-4", ci, False))


### BAAI/bge-large-en-v1.5 (local, open-weight -- REAL, no API key)

In [7]:
local_embedder = LocalEmbedder()
ci = score_embedder(local_embedder, "BAAI/bge-large-en-v1.5")
results.append(("bge-large-en-v1.5 (REAL)", ci, True))


E:\Rajesh\PycharmProjects\rag-recipes\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


E:\Rajesh\PycharmProjects\rag-recipes\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rajesh Mane\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:  24%|██▍       | 94/391 [00:00<00:00, 904.61it/s]

Loading weights:  72%|███████▏  | 280/391 [00:00<00:00, 1421.97it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1656.39it/s]

## Results

In [8]:
print_results_table(results)

variant                      paper_hit@10             status
text-embedding-3-small       0.889  [95% CI 0.722, 1.000] PENDING (mock)
voyage-4                     0.889  [95% CI 0.722, 1.000] PENDING (mock)
bge-large-en-v1.5 (REAL)     0.944  [95% CI 0.833, 1.000] REAL


## Where this study is incomplete

**PENDING: text-embedding-3-small and voyage-4 rows.** Both need a real API key
(`OPENAI_API_KEY`/`VOYAGE_API_KEY`) to reflect real retrieval quality -- under mock, both use
`MockEmbedder`, so their `paper_hit@10` numbers only prove the code path runs. The `bge-large-en-v1.5`
row above is the one genuinely real result in this notebook. See `tasks/todo.md` for the
consolidated real-key-run backlog.
